# Ndimi: fine-tune speech-to-text for one language

This notebook teaches an open speech-recognition model (OpenAI's **Whisper**) to transcribe one Kenyan language from your dataset, then tells you whether it beats what Ndimi uses today.

**Before you start**
1. Run `python ml/data_check.py /path/to/dataset` on your computer. It writes `manifest.jsonl` and `inventory.md`.
2. Upload the **dataset folder** and the **`manifest.jsonl`** to Google Drive (for example `MyDrive/ndimi/`).
3. In Colab: **Runtime → Change runtime type → GPU**. A T4 works; an L4 or A100 is faster.

**What you get**
- Word error rate (WER) on held-out speakers **before** and **after** training: lower is better.
- The trained model saved to Drive, ready for `ml/model_service`.
- Optionally, the same test scored with ElevenLabs Scribe, so you only switch Ndimi over when your model wins.

Training saves checkpoints to Drive, so if Colab disconnects, run the notebook again and it carries on from the last checkpoint.

## 1. Settings
Change these, then run every cell from top to bottom (**Runtime → Run all**).

In [ ]:
LANGUAGE = "luo"            # Ndimi code: sw, ki, luo, kln, mas, so, luy, mer, kam, guz
DIALECT = None              # e.g. "Nandi" to train on one dialect only; None = all dialects

DRIVE = "/content/drive/MyDrive/ndimi"
MANIFEST = f"{DRIVE}/out/manifest.jsonl"      # written by ml/data_check.py
AUDIO_ROOT = f"{DRIVE}/dataset"               # the folder data_check.py scanned
OUTPUT_DIR = f"{DRIVE}/models/whisper-{LANGUAGE}"

BASE_MODEL = "openai/whisper-small"   # "openai/whisper-base" is faster; "openai/whisper-medium" is better but needs an A100
MAX_STEPS = 4000        # about 1–3 hours on a T4 for whisper-small; lower it for a quick first try
BATCH_SIZE = 16         # lower to 8 if you run out of GPU memory
LEARNING_RATE = 1e-5
EVAL_EVERY = 500        # steps between checks on the dev set (and checkpoints)
MAX_TEST_CLIPS = 500    # clips used to compare models (more = slower but steadier numbers)

# Whisper only knows some languages. For the others we borrow the closest one it knows as a
# starting point; training then teaches it the real language.
WHISPER_LANGUAGE = {"sw": "swahili", "so": "somali"}.get(LANGUAGE, "swahili")

HF_REPO = None          # e.g. "jeotamedia/whisper-small-luo" to also upload a private copy to Hugging Face
ELEVENLABS_API_KEY = "" # optional: paste a key to score ElevenLabs Scribe on the same test clips

## 2. Install and connect Google Drive

In [ ]:
!pip -q install "transformers>=4.46,<5" "accelerate>=0.34" jiwer librosa soundfile
from google.colab import drive
drive.mount("/content/drive")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none. Switch the runtime to GPU first.")

## 3. Load this language's clips
Uses only clips `data_check.py` marked usable, and keeps its speaker-based split, so test speakers are never heard during training.

In [ ]:
import json, os, random, re, unicodedata, collections

rows = [json.loads(l) for l in open(MANIFEST, encoding="utf-8") if l.strip()]
rows = [r for r in rows if r["language"] == LANGUAGE and r["usable_for_asr"] and r["audio"]]
if DIALECT:
    rows = [r for r in rows if (r.get("dialect") or "").lower() == DIALECT.lower()]
for r in rows:
    r["path"] = os.path.join(AUDIO_ROOT, r["audio"])
missing = [r for r in rows if not os.path.exists(r["path"])]
if missing:
    print(f"{len(missing)} audio files not found under AUDIO_ROOT, e.g. {missing[0]['path']}. Check AUDIO_ROOT.")
rows = [r for r in rows if os.path.exists(r["path"])]
assert rows, "No usable clips for this language. Check LANGUAGE, MANIFEST and AUDIO_ROOT."

split = collections.defaultdict(list)
for r in rows:
    split[r["split"]].append(r)
if not split["dev"] or not split["test"]:
    print("Too few speakers for a speaker-based split; splitting by clip instead (scores will look a bit better than reality).")
    random.Random(0).shuffle(rows)
    n = len(rows)
    split = {"test": rows[: n // 10 or 1], "dev": rows[n // 10 or 1: 2 * (n // 10) or 2], "train": rows[2 * (n // 10) or 2:]}

hours = lambda rs: sum(r["duration"] or 0 for r in rs) / 3600
for name in ("train", "dev", "test"):
    print(f"{name:5s} {len(split[name]):6,} clips  {hours(split[name]):6.2f} h  {len({r['speaker'] for r in split[name]}):4} speakers")

random.Random(1).shuffle(split["test"])
test_rows = split["test"][:MAX_TEST_CLIPS]
dev_rows = split["dev"][:300]

def normalize(text):
    """How transcripts are compared for WER: lowercase, no punctuation, accents kept (ĩ, ũ matter)."""
    text = unicodedata.normalize("NFC", text or "").lower()
    text = re.sub(r"[^\w\s'’-]", " ", text)
    text = re.sub(r"(?<!\w)['’-]|['’-](?!\w)", " ", text)
    return re.sub(r"\s+", " ", text).strip()

## 4. Load the base model

In [ ]:
import librosa, jiwer, numpy as np
from dataclasses import dataclass
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer)

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language=WHISPER_LANGUAGE, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.generation_config.language = WHISPER_LANGUAGE
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.config.use_cache = False          # needed for gradient checkpointing
device = "cuda" if torch.cuda.is_available() else "cpu"

def load_audio(path):
    audio, _ = librosa.load(path, sr=16000, mono=True)
    return audio

def transcribe(m, rs, batch=16):
    m.eval(); out = []
    for i in range(0, len(rs), batch):
        chunk = rs[i:i + batch]
        feats = processor.feature_extractor([load_audio(r["path"]) for r in chunk], sampling_rate=16000,
                                            return_tensors="pt").input_features.to(m.device, m.dtype)
        with torch.no_grad():
            ids = m.generate(feats, language=WHISPER_LANGUAGE, task="transcribe", max_new_tokens=225)
        out += processor.batch_decode(ids, skip_special_tokens=True)
    return out

def wer(rs, hyps):
    pairs = [(normalize(r["text"]), normalize(h)) for r, h in zip(rs, hyps)]
    pairs = [(r, h) for r, h in pairs if r]
    return 100 * jiwer.wer([r for r, _ in pairs], [h for _, h in pairs])

## 5. Score the model before training
This is the starting point. For languages Whisper has never seen, expect a very high WER (often above 90%).

In [ ]:
model.to(device)
baseline_hyps = transcribe(model, test_rows)
baseline_wer = wer(test_rows, baseline_hyps)
print(f"Before training: WER {baseline_wer:.1f}% on {len(test_rows)} test clips")
for r, h in list(zip(test_rows, baseline_hyps))[:3]:
    print("  reference:", r["text"]); print("  model:    ", h); print()

## 6. Train
Checkpoints are saved to Drive every `EVAL_EVERY` steps. The dev-set WER printed during training should fall.

In [ ]:
class Clips(torch.utils.data.Dataset):
    def __init__(self, rs): self.rs = rs
    def __len__(self): return len(self.rs)
    def __getitem__(self, i):
        r = self.rs[i]
        feats = processor.feature_extractor(load_audio(r["path"]), sampling_rate=16000).input_features[0]
        return {"input_features": feats, "labels": processor.tokenizer(r["text"]).input_ids}

@dataclass
class Collator:
    def __call__(self, items):
        batch = processor.feature_extractor.pad([{"input_features": x["input_features"]} for x in items], return_tensors="pt")
        labels = processor.tokenizer.pad([{"input_ids": x["labels"]} for x in items], return_tensors="pt")
        ids = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        if (ids[:, 0] == model.config.decoder_start_token_id).all():
            ids = ids[:, 1:]
        batch["labels"] = ids
        return batch

def compute_metrics(pred):
    labels = pred.label_ids.copy()
    labels[labels == -100] = processor.tokenizer.pad_token_id
    hyps = processor.batch_decode(pred.predictions, skip_special_tokens=True)
    refs = processor.batch_decode(labels, skip_special_tokens=True)
    pairs = [(normalize(r), normalize(h)) for r, h in zip(refs, hyps) if normalize(r)]
    return {"wer": 100 * jiwer.wer([r for r, _ in pairs], [h for _, h in pairs])}

args = Seq2SeqTrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=max(1, 16 // BATCH_SIZE),
    learning_rate=LEARNING_RATE,
    warmup_steps=min(500, MAX_STEPS // 10),
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps", eval_steps=EVAL_EVERY,
    save_strategy="steps", save_steps=EVAL_EVERY, save_total_limit=2,
    per_device_eval_batch_size=8,
    predict_with_generate=True, generation_max_length=225,
    load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
    logging_steps=25, report_to="none", dataloader_num_workers=2, remove_unused_columns=False,
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=Clips(split["train"]), eval_dataset=Clips(dev_rows),
                         data_collator=Collator(), compute_metrics=compute_metrics,
                         processing_class=processor.feature_extractor)

ckpt_dir = f"{OUTPUT_DIR}/checkpoints"
resume = os.path.isdir(ckpt_dir) and any(d.startswith("checkpoint-") for d in os.listdir(ckpt_dir))
print("Resuming from the last checkpoint." if resume else "Starting a new run.")
trainer.train(resume_from_checkpoint=resume or None)

## 7. Score the trained model on the same test clips

In [ ]:
model = trainer.model
model.config.use_cache = True
tuned_hyps = transcribe(model, test_rows)
tuned_wer = wer(test_rows, tuned_hyps)
print(f"Before training: WER {baseline_wer:.1f}%")
print(f"After training:  WER {tuned_wer:.1f}%   (lower is better)")
for r, h in list(zip(test_rows, tuned_hyps))[:5]:
    print("  reference:", r["text"]); print("  model:    ", h); print()

## 8. Optional: compare with ElevenLabs Scribe
Ndimi uses Scribe today. Switch a language to your own model only if your WER is clearly lower. Scoring costs ElevenLabs credits (roughly the length of the test clips).

In [ ]:
import requests
scribe_wer = None
if ELEVENLABS_API_KEY:
    hyps = []
    for r in test_rows:
        with open(r["path"], "rb") as fh:
            resp = requests.post("https://api.elevenlabs.io/v1/speech-to-text",
                                 headers={"xi-api-key": ELEVENLABS_API_KEY},
                                 data={"model_id": "scribe_v2"}, files={"file": fh}, timeout=120)
        hyps.append(resp.json().get("text", "") if resp.ok else "")
    scribe_wer = wer(test_rows, hyps)
    print(f"ElevenLabs Scribe: WER {scribe_wer:.1f}%   Your model: {tuned_wer:.1f}%")
    print("Your model wins. Switch this language to Ndimi models." if tuned_wer < scribe_wer
          else "Scribe is still better. Keep it for now, add data or train longer.")
else:
    print("Skipped (no ElevenLabs key).")

## 9. Save the model
Saves to `OUTPUT_DIR` on Drive, plus `results.json` with the scores. Point `ml/model_service/models.json` at this folder (or the Hugging Face repo).

In [ ]:
import datetime
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
model.generation_config.save_pretrained(OUTPUT_DIR)
results = {
    "language": LANGUAGE, "dialect": DIALECT, "base_model": BASE_MODEL, "whisper_language_token": WHISPER_LANGUAGE,
    "train_hours": round(hours(split["train"]), 2), "train_clips": len(split["train"]),
    "test_clips": len(test_rows), "steps": MAX_STEPS,
    "wer_before": round(baseline_wer, 2), "wer_after": round(tuned_wer, 2),
    "wer_elevenlabs_scribe": None if scribe_wer is None else round(scribe_wer, 2),
    "trained_at": datetime.datetime.utcnow().isoformat() + "Z",
}
json.dump(results, open(f"{OUTPUT_DIR}/results.json", "w"), indent=2)
print(json.dumps(results, indent=2))
print("\nAdd this to ml/model_service/models.json:")
print(json.dumps({"transcribe": {LANGUAGE: {"model": HF_REPO or OUTPUT_DIR, "language_token": WHISPER_LANGUAGE}}}, indent=2))

if HF_REPO:
    model.push_to_hub(HF_REPO, private=True)
    processor.push_to_hub(HF_REPO, private=True)
    print("Uploaded to https://huggingface.co/" + HF_REPO)

## 10. Try it on any recording

In [ ]:
from google.colab import files
uploaded = files.upload()   # pick an audio file from your computer
for name in uploaded:
    print(name, "→", transcribe(model, [{"path": name}])[0])